# 02 — Photometric augmentation (focus: face + license-plate)

From each 1280×1280 crop produced in phase 1 we generate **1 original + N photometric
variants**. The factor is class-dependent:

- crops with ≥1 `face` OR `license-plate` → **20×** (`n_variants = 19`)
- all other crops → **10×** (`n_variants = 9`)

This enriches the two target classes without inflating the dataset unnecessarily.

### Why photometric only?
Geometric augmentation (rotation, anchored crops) was already done in phase 1. Here we
change **pixel values only** (brightness, noise, blur, fog, simulated compression
artifacts). The **bounding boxes are unchanged**, so annotations are copied with a new
`image_id`. Output is saved **losslessly as PNG** (the ImageCompression transform is only
a pixel effect; the files themselves are never lossy-compressed).

Reproducible via a fixed seed (42). The worker `scripts/photometric_worker.py` runs in
parallel (ProcessPoolExecutor) and is resumable. Output: `aug/images/*.png` +
`aug/annotations_coco.json`.

## 1. Config & load crop annotations

In [ ]:
import os, json, random, copy
import numpy as np
import cv2
from collections import Counter
from tqdm.auto import tqdm

ROOT = "/home/jovyan/shared/s0598584"
CROP_DIR = os.path.join(ROOT, "crops_1280")
AUG_DIR  = os.path.join(ROOT, "aug")
AUG_IMG  = os.path.join(AUG_DIR, "images")
os.makedirs(AUG_IMG, exist_ok=True)

FACTOR_FOCUS = 20   # face/license-plate -> 20x (1 original + 19 variants)
FACTOR_REST  = 10   # everything else -> 10x
SEED = 42
random.seed(SEED); np.random.seed(SEED)

with open(os.path.join(CROP_DIR, "annotations_coco.json")) as f:
    crop_coco = json.load(f)

images     = crop_coco["images"]
categories = crop_coco["categories"]
name2id = {c["name"]: c["id"] for c in categories}
FACE_ID = name2id["face"]; LP_ID = name2id["license-plate"]

anns_by_img = {}
for a in crop_coco["annotations"]:
    anns_by_img.setdefault(a["image_id"], []).append(a)

# Per crop: does it contain a focus class (face/lp)?
focus_img_ids = set()
for im in images:
    cats = {a["category_id"] for a in anns_by_img.get(im["id"], [])}
    if FACE_ID in cats or LP_ID in cats:
        focus_img_ids.add(im["id"])

n_focus = len(focus_img_ids); n_rest = len(images) - n_focus
est_tiles = n_focus * FACTOR_FOCUS + n_rest * FACTOR_REST
print(f"source crops: {len(images)} | annotations: {len(crop_coco['annotations'])}")
print(f"  focus (face/lp) : {n_focus}  -> {FACTOR_FOCUS}x")
print(f"  rest            : {n_rest}  -> {FACTOR_REST}x")
print(f"  expected tiles  : {est_tiles}")

## 2. Load the worker module

In [ ]:
import sys
sys.path.insert(0, os.path.join(ROOT, "scripts"))
import photometric_worker as pw
_t = pw.build_transform()
print("worker loaded. transform stages:", len(_t.transforms), "| PNG params:", pw._PNG_PARAMS)

## 3. Augment (parallel) & build COCO

Each task gets `n_variants` according to its focus status and a deterministic seed
(`SEED + index`). Filenames keep the source stem as a prefix — important for the
leakage-free split in phase 3. The COCO annotations are built deterministically in the
main process (one copy per variant v0..vN).

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

images_sorted = sorted(images, key=lambda im: im["file_name"])

def nvar_for(im):
    return (FACTOR_FOCUS - 1) if im["id"] in focus_img_ids else (FACTOR_REST - 1)

tasks = []
for i, im in enumerate(images_sorted):
    stem = os.path.splitext(im["file_name"])[0]
    tasks.append({
        "src_path": os.path.join(CROP_DIR, "images", im["file_name"]),
        "stem": stem,
        "out_dir": AUG_IMG,
        "n_variants": nvar_for(im),
        "seed": SEED + i,
    })

ok_stems = set(); failed = []
WORKERS = 16
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(pw.process_one, t): t for t in tasks}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Crops"):
        r = fut.result()
        (ok_stems.add(r["stem"]) if r["ok"] else failed.append(r["stem"]))
print("processed crops:", len(ok_stems), "| failed:", len(failed))
if failed:
    print("failed (first 10):", failed[:10])

# Build COCO deterministically (n_variants per crop)
nvar_by_stem = {os.path.splitext(im["file_name"])[0]: nvar_for(im) for im in images_sorted}
out_images, out_anns = [], []
next_img_id = 0; next_ann_id = 0

def copy_anns(src_anns, new_img_id):
    global next_ann_id
    res = []
    for a in src_anns:
        na = copy.deepcopy(a)
        na["id"] = next_ann_id; next_ann_id += 1
        na["image_id"] = new_img_id
        res.append(na)
    return res

for im in images_sorted:
    stem = os.path.splitext(im["file_name"])[0]
    if stem not in ok_stems:
        continue
    src_anns = anns_by_img.get(im["id"], [])
    nv = nvar_by_stem[stem]
    for v in range(0, nv + 1):
        out_name = f"{stem}_v{v}.png"
        out_images.append({"id": next_img_id, "file_name": out_name, "width": 1280, "height": 1280})
        out_anns.extend(copy_anns(src_anns, next_img_id))
        next_img_id += 1

print("COCO images:", len(out_images), "| COCO annotations:", len(out_anns))

## 4. Write the augmented COCO JSON

In [ ]:
aug_coco = {"images": out_images, "annotations": out_anns, "categories": categories}
ann_out = os.path.join(AUG_DIR, "annotations_coco.json")
with open(ann_out, "w") as f:
    json.dump(aug_coco, f)
print("written:", ann_out)

## 5. Verify: image count, label distribution, example shape

In [ ]:
import glob
pngs = glob.glob(os.path.join(AUG_IMG, "*.png"))
print("PNGs on disk    :", len(pngs))
print("COCO images     :", len(aug_coco["images"]))
print("COCO annotations:", len(aug_coco["annotations"]))
assert len(pngs) == len(aug_coco["images"])

cat = {c["id"]: c["name"] for c in categories}
dist = Counter(a["category_id"] for a in aug_coco["annotations"])
tot = sum(dist.values())
print("\nLabel distribution (aug):")
for cid, n in dist.most_common():
    print(f"  {cat[cid]:16s} {n:7d}  ({n/tot*100:5.1f}%)")

im = cv2.imread(pngs[0]); print("\nExample shape:", im.shape)
assert im.shape[:2] == (1280, 1280)

## 6. Disk budget after augmentation

In [ ]:
import shutil
aug_bytes = sum(os.path.getsize(p) for p in pngs)
total, used, free = shutil.disk_usage("/home/jovyan/shared")
print(f"aug/ size       : {aug_bytes/1e9:.1f} GB ({len(pngs)} PNGs, ~{aug_bytes/len(pngs)/1e6:.2f} MB/tile)")
print(f"shared free     : {free/1e9:.0f} GB")
assert free/1e9 > 15, "buffer < 15 GB — reduce the factor!"
print("disk buffer OK.")

## ✅ Phase 2 done

Lossless PNG tiles (20× for face/lp, 10× otherwise) + `aug/annotations_coco.json`.
**Continue with `03_coco_to_yolo_split.ipynb`.**